In [2]:
# imports
import collections
import contextlib
import sys
import wave
# !pip install webrtcvad
# import webrtcvad
import audioop
# !pip install pylangacq
import glob
import os
import math
import time

import re
import csv
import pylangacq
import numpy as np
import pandas as pd
from pathlib import Path
import contextlib
import re
import wave
# !pip install mutagen
from mutagen.mp3 import MP3

import numpy as np
np.random.seed(0)
p = np.random.permutation(108) # n_samples = 108
p_subjects = np.random.RandomState(seed=0).permutation(242)
# import tensorflow as tf
# from tensorflow.keras import layers
# from tensorflow.keras.models import Model
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression


# Data processing

## Interpetable features extraction

Collecting files 

we can make this a class and add a function to save preprocesed files but is it really needed? maybe not, we can just save time

In [9]:
cha_path = Path(r"D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha")

In [14]:
from pathlib import Path
import pandas as pd

# roots
wav_root = Path("wav")
cha_root = Path("cha")

# map class folder name -> label
# folder structure is: Pitt-cha/dementia/cha, Pitt-cha/dementia/wav, Pitt-cha/control/cha, Pitt-cha/control/wav
label_map = {
    "dementia": 1,
    "control": 0,
}

rows = []

for class_name, label in label_map.items():
    # merge paths
    wav_dir = cha_path / class_name / wav_root 
    cha_dir = cha_path / class_name / cha_root 
    print(wav_dir, cha_dir)

    # collect files by stem
    wav_files = {p.stem: p for p in wav_dir.glob("*.wav")}
    cha_files = {p.stem: p for p in cha_dir.glob("*.cha")}
    print(f"Collected files, wav: {len(wav_files)}, cha: {len(cha_files)}")

    # only keep matched pairs
    common_ids = sorted(set(wav_files) & set(cha_files))
    print("Sorted ids")
    print(len(common_ids))
    print(common_ids[:5])

    # optional: report unmatched files
    missing_wav = sorted(set(cha_files) - set(wav_files))
    missing_cha = sorted(set(wav_files) - set(cha_files))

    if missing_wav:
        print(f"[WARN] {class_name}: {len(missing_wav)} .cha files without matching .wav")
        print(missing_wav[:10])

    if missing_cha:
        print(f"[WARN] {class_name}: {len(missing_cha)} .wav files without matching .cha")
        print(missing_cha[:10])

    for file_id in common_ids:
        rows.append({
            "file_id": file_id,
            "wav_path": str(wav_files[file_id]),
            "cha_path": str(cha_files[file_id]),
            "label": label,
        })

files = pd.DataFrame(rows)

print(files.head())
print()
print("Shape:", files.shape)
print(files["label"].value_counts())

# optional save
# df.to_csv("pitt_cookie_file_index.csv", index=False)

D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\dementia\wav D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\dementia\cha
Collected files, wav: 180, cha: 309
Sorted ids
180
['016-0', '050-0', '062-3', '076-0', '076-4']
[WARN] dementia: 129 .cha files without matching .wav
['001-0', '001-2', '003-0', '005-0', '005-2', '007-1', '007-3', '010-0', '010-1', '010-2']
D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\control\wav D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\control\cha
Collected files, wav: 189, cha: 243
Sorted ids
189
['002-3', '015-2', '021-1', '021-2', '021-3']
[WARN] control: 54 .cha files without matching .wav
['002-0', '002-1', '002-2', '006-2', '006-3', '006-4', '013-0', '013-2', '013-3', '013-4']
  file_id                                           wav_path  \
0   016-0  D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...   
1   050-0  D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...   
2   062-3  D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...   
3   076-0  D:\masteruwefduyq

In [15]:
files

,file_id,wav_path,cha_path,label
0,016-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1
1,050-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1
2,062-3,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1
3,076-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1
4,076-4,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1
...,...,...,...,...
364,686-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0
365,688-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0
366,691-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0
367,709-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0


In [ ]:
# preprocess

In [3]:
# import pandas as pd
import pandas as pd

files = pd.DataFrame({
    "file_id": ["002-3"],
    "wav_path": [r"C:\Users\DELL\Downloads\002-3.wav"],
    "cha_path": [r"C:\Users\DELL\Downloads\Pitt\Pitt\Control\cookie\002-3.cha"],
    "label": [0],
})

print(files)

  file_id                           wav_path  \
0   002-3  C:\Users\DELL\Downloads\002-3.wav   

                                            cha_path  label  
0  C:\Users\DELL\Downloads\Pitt\Pitt\Control\cook...      0  


In [16]:
# for each line collect features
# total duration speaking normalized by total duration of recording
# interviewer interruptions
# how many times they stopped
import pylangacq
feature_rows = []
for index, row in files.iterrows():
    print(row)
    wav_file = row['wav_path']
    cha_file = row["cha_path"]
    label = row["label"]
    print(wav_file, cha_file, label)
    # computing interpretable features from .cha files
    # 1. total duration of recording
    # 2. total duration of subject speaking
    try:
        reader = pylangacq.read_chat(cha_file)
    except Exception as e:
        print(f"Could not read {cha_file}: {e}")
        continue
    
    rows_cha = []

    for utt in reader.utterances():
        text = " ".join(token.word for token in utt.tokens)
        time_marks = utt.time_marks
        start_time = time_marks[0] if time_marks is not None else np.nan
        end_time = time_marks[1] if time_marks is not None else np.nan
        rows_cha.append({
            "speaker": utt.participant,
            "text": text,
            "time": utt.time_marks,
            "start_time": start_time,
            "end_time": end_time,

        })


    cha = pd.DataFrame(rows_cha)
    if cha.empty:
        print(f"Empty transcript: {cha_file}")
        continue
    print(cha)

    # 1. total speaking duration
    # getting total speaking miliseconds of par from the table
    cha["duration"] = cha["time"].apply(
    lambda x: x[1] - x[0] if x is not None else None
    )
    total_speaking_time = cha[cha["speaker"] == "PAR"]["duration"].sum()
    total_time = cha["time"].apply(lambda x: x[1] if x is not None else None).max()
    print(f"Total speaking time: {total_speaking_time:.2f} ms, Total time: {total_time:.2f} ms")
    total_speaking_time_normalized = total_speaking_time / total_time if total_time > 0 else 0
    print(f"Total speaking time normalized: {total_speaking_time_normalized:.4f}")

    # 2. interviewer interruptions
    # count how many times the interviewer (INV) interrupts the subject (PAR)
    cha["is_par"] = cha["speaker"] == "PAR"
    cha["is_inv"] = cha["speaker"] == "INV"
    cha["par_end_time"] = cha.apply(lambda row: row["time"][1] if row["time"] is not None else None if row["is_par"] else None, axis=1)
    cha["inv_start_time"] = cha.apply(lambda row: row["time"][0] if row["time"] is not None else None if row["is_inv"] else None, axis=1)
    # this below might be wrong check
    cha["inv_interrupts_par"] = cha.apply(lambda row: 1 if row["is_inv"] and any((cha["is_par"] & (cha["par_end_time"] > row["inv_start_time"])) ) else 0, axis=1)
    interviewer_interruptions = cha["inv_interrupts_par"].sum()
    print(f"Interviewer interruptions: {interviewer_interruptions}")

    # 3. pause features
    # we will see if it is needed

    # 4. word rate
    # count the number of words spoken by the subject
    cha["word_count"] = cha["text"].apply(lambda x: len(x.split()) if x is not None else 0)
    word_count = cha[cha["speaker"] == "PAR"]["word_count"].sum()
    print(f"Word count: {word_count}")
    word_rate = word_count / (total_speaking_time / 1000) if total_speaking_time > 0 else 0
    print(f"Word rate: {word_rate:.2f} words/sec")

    # 5. pauses as filler words etc
    feature_rows.append({
            "wav_path": wav_file,
            "cha_path": cha_file,
            "label": label,
            "total_time_ms": total_time,
            "total_speaking_time_ms": total_speaking_time,
            "total_speaking_time_normalized": total_speaking_time_normalized,
            "interviewer_interruptions": interviewer_interruptions,
            # "interruption_rate_per_sec": interruption_rate,
            "word_count": word_count,
            "word_rate_words_per_sec": word_rate,
            # "pause_count": pause_count,
            # "pause_rate_per_sec": pause_rate,
            # "total_pause_duration_ms": total_pause_duration,
            # "mean_pause_duration_ms": mean_pause_duration
        })
    

file_id                                                 016-0
wav_path    D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...
cha_path    D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...
label                                                       1
Name: 0, dtype: object
D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\dementia\wav\016-0.wav D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\dementia\cha\016-0.cha 1
  speaker                                               text            time  \
0     PAR  guy or a boy reaching for a cookie jar tilting...   (3490, 10186)   
1     PAR  and a girl asking for one pointing to her mouth .  (10841, 15161)   
2     INV                                                  .            None   
3     PAR                            a woman drying dishes .  (16328, 20914)   
4     PAR  outdoors I don't see anything going on but the...  (31529, 46048)   
5     PAR  and there's a tree which you can see through t...  (48459, 51819)   
6     INV                    

TypeError: 'NoneType' object is not iterable

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import librosa


# =========================
# Configuration
# =========================

WAV_ROOT = Path("wav")
CHA_ROOT = Path("cha")
OUTPUT_CSV = Path("features_cookie_pitt.csv")

# label mapping inferred from folder names
POSITIVE_NAMES = {"dementia", "ad", "alzheimers", "alzheimer"}
NEGATIVE_NAMES = {"non dementia", "non_dementia", "control", "controls", "healthy", "hc"}

SR = 16000
N_MFCC = 13


# =========================
# Utility functions
# =========================

def normalize_label_name(name: str) -> str:
    return name.strip().lower().replace("-", " ").replace("_", " ")


def infer_label_from_folder(folder_name: str) -> int:
    name = normalize_label_name(folder_name)
    if name in POSITIVE_NAMES:
        return 1
    if name in NEGATIVE_NAMES:
        return 0
    raise ValueError(f"Unrecognized class folder name: {folder_name!r}")


def find_files_by_class(root: Path, suffix: str) -> Dict[Tuple[str, str], Path]:
    """
    Returns mapping: (class_folder_name, stem) -> file_path
    Example key: ("dementia", "123-0")
    """
    mapping = {}
    for class_dir in root.iterdir():
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        for f in class_dir.rglob(f"*{suffix}"):
            mapping[(class_name, f.stem)] = f
    return mapping


# =========================
# CHAT / CHA parsing
# =========================

CHAT_BRACKET_RE = re.compile(r"\[[^\]]*\]")
CHAT_EVENT_RE = re.compile(r"&=\w+")
MULTISPACE_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"[a-zA-Z']+")

# Some very rough markers often useful in CHAT-like transcripts
FILLER_RE = re.compile(r"\b(um|uh|erm|mm|hmm)\b", re.IGNORECASE)

def load_cha_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def extract_par_lines(cha_text: str) -> List[str]:
    """
    Keep only participant lines: *PAR:
    Continuation lines are ignored in this simple baseline.
    """
    lines = cha_text.splitlines()
    par_lines = []
    for line in lines:
        if line.startswith("*PAR:"):
            par_lines.append(line[len("*PAR:"):].strip())
    return par_lines


def clean_chat_for_lexical(line: str) -> str:
    """
    Simplified CHAT cleaning for lexical counts.
    This is intentionally conservative and not a full CHAT parser.
    """
    line = CHAT_BRACKET_RE.sub(" ", line)     # remove [//], [*], etc.
    line = CHAT_EVENT_RE.sub(" ", line)       # remove &=laughs style events
    line = line.replace("<", " ").replace(">", " ")
    line = re.sub(r"\(\.\)|\(\.\.\)|\(\.\.\.\)", " ", line)  # pause notation if present
    line = re.sub(r"[^\w'\s-]", " ", line)    # remove punctuation-ish symbols
    line = MULTISPACE_RE.sub(" ", line).strip().lower()
    return line


def tokenize_words(text: str) -> List[str]:
    return WORD_RE.findall(text.lower())


def extract_text_features(cha_path: Path) -> Dict[str, float]:
    cha_text = load_cha_text(cha_path)
    par_lines = extract_par_lines(cha_text)

    raw_joined = " ".join(par_lines)
    cleaned_lines = [clean_chat_for_lexical(x) for x in par_lines]
    cleaned_text = " ".join(cleaned_lines)
    tokens = tokenize_words(cleaned_text)

    num_words = len(tokens)
    unique_words = len(set(tokens))
    utt_lengths = [len(tokenize_words(x)) for x in cleaned_lines if x.strip()]
    mean_words_per_utt = float(np.mean(utt_lengths)) if utt_lengths else 0.0
    std_words_per_utt = float(np.std(utt_lengths)) if utt_lengths else 0.0

    # very rough disfluency proxies
    fillers = FILLER_RE.findall(raw_joined)
    repetitions_marker = len(re.findall(r"\[/\]|\[//\]", cha_text))
    retracing_marker = len(re.findall(r"\[//\]", cha_text))
    error_marker = len(re.findall(r"\[\*\]", cha_text))
    fragment_like = len(re.findall(r"\b\w+\-\b", raw_joined))

    word_counts = Counter(tokens)
    avg_word_len = float(np.mean([len(w) for w in tokens])) if tokens else 0.0

    # Cookie Theft content words: simple task-specific feature
    cookie_words = {
        "boy", "girl", "mother", "woman", "cookie", "cookies", "jar",
        "sink", "water", "stool", "window", "plate", "dish", "kitchen"
    }
    cookie_keyword_hits = sum(1 for w in cookie_words if w in word_counts)
    cookie_keyword_coverage = cookie_keyword_hits / len(cookie_words)

    return {
        "num_utterances": float(len(par_lines)),
        "num_words": float(num_words),
        "num_unique_words": float(unique_words),
        "type_token_ratio": float(unique_words / num_words) if num_words else 0.0,
        "avg_word_len": avg_word_len,
        "mean_words_per_utt": mean_words_per_utt,
        "std_words_per_utt": std_words_per_utt,
        "fillers_um_uh": float(len(fillers)),
        "repetitions_marker": float(repetitions_marker),
        "retracing_marker": float(retracing_marker),
        "error_marker": float(error_marker),
        "fragment_like": float(fragment_like),
        "cookie_keyword_coverage": float(cookie_keyword_coverage),
    }


# =========================
# Acoustic features
# =========================

def extract_pause_features(y: np.ndarray, sr: int, top_db: int = 30, min_pause_sec: float = 0.2) -> Dict[str, float]:
    """
    Simple silence-based pause estimate using librosa.effects.split.
    """
    total_dur = len(y) / sr
    intervals = librosa.effects.split(y, top_db=top_db)

    if len(intervals) == 0:
        return {
            "duration_sec": float(total_dur),
            "speech_time_sec": 0.0,
            "pause_time_sec": float(total_dur),
            "pause_ratio": 1.0 if total_dur > 0 else 0.0,
            "num_pauses": 1.0 if total_dur >= min_pause_sec else 0.0,
            "mean_pause_sec": float(total_dur),
            "max_pause_sec": float(total_dur),
        }

    speech_segments = [(s / sr, e / sr) for s, e in intervals]
    speech_time = sum(e - s for s, e in speech_segments)

    pauses = []
    prev_end = 0.0
    for s, e in speech_segments:
        gap = s - prev_end
        if gap >= min_pause_sec:
            pauses.append(gap)
        prev_end = e

    final_gap = total_dur - prev_end
    if final_gap >= min_pause_sec:
        pauses.append(final_gap)

    pause_time = sum(pauses)

    return {
        "duration_sec": float(total_dur),
        "speech_time_sec": float(speech_time),
        "pause_time_sec": float(pause_time),
        "pause_ratio": float(pause_time / total_dur) if total_dur > 0 else 0.0,
        "num_pauses": float(len(pauses)),
        "mean_pause_sec": float(np.mean(pauses)) if pauses else 0.0,
        "max_pause_sec": float(np.max(pauses)) if pauses else 0.0,
    }


def extract_audio_features(wav_path: Path, sr: int = SR, n_mfcc: int = N_MFCC) -> Dict[str, float]:
    y, sr = librosa.load(wav_path, sr=sr)

    feats = {}

    # basic frame-level descriptors
    rms = librosa.feature.rms(y=y).flatten()
    zcr = librosa.feature.zero_crossing_rate(y).flatten()
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr).flatten()
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr).flatten()
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr).flatten()

    feats["rms_mean"] = float(np.mean(rms))
    feats["rms_std"] = float(np.std(rms))
    feats["zcr_mean"] = float(np.mean(zcr))
    feats["zcr_std"] = float(np.std(zcr))
    feats["spec_centroid_mean"] = float(np.mean(centroid))
    feats["spec_centroid_std"] = float(np.std(centroid))
    feats["spec_bandwidth_mean"] = float(np.mean(bandwidth))
    feats["spec_bandwidth_std"] = float(np.std(bandwidth))
    feats["rolloff_mean"] = float(np.mean(rolloff))
    feats["rolloff_std"] = float(np.std(rolloff))

    # MFCCs -> summarize across time
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        feats[f"mfcc_{i+1}_mean"] = float(np.mean(mfcc[i]))
        feats[f"mfcc_{i+1}_std"] = float(np.std(mfcc[i]))

    # crude pitch estimate
    pitches, mags = librosa.piptrack(y=y, sr=sr)
    pitch_vals = pitches[mags > np.median(mags)]
    pitch_vals = pitch_vals[pitch_vals > 0]

    feats["f0_mean"] = float(np.mean(pitch_vals)) if len(pitch_vals) else 0.0
    feats["f0_std"] = float(np.std(pitch_vals)) if len(pitch_vals) else 0.0
    feats["f0_min"] = float(np.min(pitch_vals)) if len(pitch_vals) else 0.0
    feats["f0_max"] = float(np.max(pitch_vals)) if len(pitch_vals) else 0.0

    feats.update(extract_pause_features(y, sr))
    return feats


# =========================
# Pairing and dataset build
# =========================

def build_pairs(wav_root: Path, cha_root: Path) -> List[Dict[str, object]]:
    wav_map = find_files_by_class(wav_root, ".wav")
    cha_map = find_files_by_class(cha_root, ".cha")

    common_keys = sorted(set(wav_map.keys()) & set(cha_map.keys()))
    missing_wav = sorted(set(cha_map.keys()) - set(wav_map.keys()))
    missing_cha = sorted(set(wav_map.keys()) - set(cha_map.keys()))

    if missing_wav:
        print(f"[WARN] {len(missing_wav)} transcript files have no matching wav.")
        for k in missing_wav[:10]:
            print("  missing wav for:", k)

    if missing_cha:
        print(f"[WARN] {len(missing_cha)} wav files have no matching transcript.")
        for k in missing_cha[:10]:
            print("  missing cha for:", k)

    pairs = []
    for class_name, stem in common_keys:
        label = infer_label_from_folder(class_name)
        pairs.append({
            "file_id": stem,
            "class_name": class_name,
            "label": label,
            "wav_path": wav_map[(class_name, stem)],
            "cha_path": cha_map[(class_name, stem)],
        })
    return pairs


def extract_record_features(record: Dict[str, object]) -> Dict[str, object]:
    wav_path: Path = record["wav_path"]  # type: ignore
    cha_path: Path = record["cha_path"]  # type: ignore

    row = {
        "file_id": record["file_id"],
        "class_name": record["class_name"],
        "label": record["label"],
        "wav_path": str(wav_path),
        "cha_path": str(cha_path),
    }

    try:
        row.update(extract_audio_features(wav_path))
    except Exception as e:
        row["audio_error"] = str(e)

    try:
        row.update(extract_text_features(cha_path))
    except Exception as e:
        row["text_error"] = str(e)

    return row


def main():
    pairs = build_pairs(WAV_ROOT, CHA_ROOT)
    print(f"Found {len(pairs)} matched wav/cha pairs.")

    rows = []
    for i, rec in enumerate(pairs, start=1):
        if i % 25 == 0 or i == len(pairs):
            print(f"Processing {i}/{len(pairs)}")
        rows.append(extract_record_features(rec))

    df = pd.DataFrame(rows)

    # optional: stable column order
    front_cols = ["file_id", "class_name", "label", "wav_path", "cha_path", "audio_error", "text_error"]
    existing_front = [c for c in front_cols if c in df.columns]
    remaining = [c for c in df.columns if c not in existing_front]
    df = df[existing_front + remaining]

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved CSV to: {OUTPUT_CSV.resolve()}")
    print(df.head())


if __name__ == "__main__":
    main()

In [ ]:
# class taken from somewhere we need to reference it, later on put in a separate file and import it

'''
Util functions for reading and extracting data and other stuff
'''


################# Utils #################

class EasyDict(dict):
    def __init__(self, *args, **kwargs): super().__init__(*args, **kwargs)
    def __getattr__(self, name): return self[name]
    def __setattr__(self, name, value): self[name] = value
    def __delattr__(self, name): del self[name]

def create_directories(config):
    model_dir = Path(config.model_dir)
    for m in config.model_types:
        model_dir.joinpath(m).mkdir(parents=True, exist_ok=True)

################# PAUSE FEATURES #################

def clean_file(lines):
    return re.sub(r'[0-9]+[_][0-9]+', '', lines.replace("*INV:", "").replace("*PAR:", "")).strip().replace("\x15", "").replace("\n", "").replace("\t", " ").replace("[+ ", "[+").replace("[* ", "[*").replace("[: ", "[:").replace(" .", "").replace("'s", "").replace(" ?", "").replace(" !", "").replace(" ]", "]").lower()

def extra_clean(lines):
    lines = clean_file(lines)
    lines = lines.replace("[+exc]", "")
    lines = lines.replace("[+gram]", "")
    lines = lines.replace("[+es]", "")
    lines = re.sub(r'[&][=]*[a-z]+', '', lines) #remove all &=text
    lines = re.sub(r'\[[*][a-z]:[a-z][-|a-z]*\]', '', lines) #remove all [*char:char(s)]
    lines = re.sub(r'[^A-Za-z0-9\s_]+', '', lines) #remove all remaining symbols except underscore
    lines = re.sub(r'[_]', ' ', lines) #replace underscore with space
    return lines

def words_count(content):
    extra_cleaned = extra_clean(content).split(" ")
    return len(extra_cleaned) - extra_cleaned.count('')

def get_pauses_cnt(content):
    content = clean_file(content)

    cnt = 0
    pauses_list = []
    pauses = re.findall(r'&[a-z]+', content) #find all utterances
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'<[a-z_\s]+>', content) #find all <text>
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\[/+\]', content) #find all [/+]
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\([\.]+\)', content) #find all (.*)
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\+[\.]+', content) #find all +...
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'[m]*hm', content) #find all mhm or hm
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\[:[a-z_\s]+\]', content) #find all [:word]
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'[a-z]*\([a-z]+\)[a-z]*', content) #find all wor(d)
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    temp = re.sub(r'\[[*][a-z]:[a-z][-|a-z]*\]', '', content)
    pauses = re.findall(r'[a-z]+:[a-z]+', temp) #find all w:ord
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    return np.array(pauses_list)


################# INTERVENTION FEATURES #################

def get_n_interventions(content):
    content = content.split('\n')
    speakers = []

    for c in content:
        if 'INV' in c:
          speakers.append('INV')
        if 'PAR' in c:
          speakers.append('PAR')

    PAR_first_index = speakers.index('PAR')
    PAR_last_index = len(speakers) - speakers[::-1].index('PAR') - 1
    speakers = speakers[PAR_first_index:PAR_last_index]
    return speakers.count('INV')


################# SPECTOGRAM FEATURES #################

def read_spectogram():
    return

################# REGRESSION FEATURES #################

def get_regression_values(metadata_filename):
    values = []
    with open(metadata_filename, 'r') as f:
        content = f.readlines()[1:]
        for idx, line in enumerate(content):
            token = line.split('; ')[-1].strip('\n')
            if token!='NA':  values.append(int(token))
            else:   values.append(30) # NA fill value

    return values

def get_classification_values(metadata_filename):
    values = []
    with open(metadata_filename, 'r') as f:
        content = f.readlines()[1:]
        for idx, line in enumerate(content):
            token = line.split('; ')[-2].strip('\n')
            if token!='NA':  values.append(int(token))
            else:   values.append(30) # NA fill value

    return values

def get_audio_length(filename):
    with contextlib.closing(wave.open(filename,'r')) as f:
        frames = f.getnframes()
        rate = f.getframerate()
        duration = frames / float(rate)
    return duration

def get_mp3_audio_length(filename):
    audio = MP3(filename)
    duration = audio.info.length
    return duration




In [ ]:
# code also taken from somewhere we need to reference it later on put in a separate file and import it


def get_pause_features(transcription_filename, audio_filename, audio_length_normalization=10):
    '''
    Pause features include word rate, pause rate of various kinds of pauses and utterances, and intervention rate
    '''
    audio_len =  get_audio_length(audio_filename)/audio_length_normalization

    with open(transcription_filename, 'r') as f:
        content = f.read()
        word_rate = words_count(content) / (50 * audio_len)
        pause_rates = get_pauses_cnt(content) / audio_len
        inv_rate = get_n_interventions(content) / audio_len

    pause_features = np.concatenate(([inv_rate], pause_rates, [word_rate]), axis=-1)

    return pause_features


def get_intervention_features(transcription_filename, max_length=40):
    '''
    Intervention features include one hot encoded sequence of speaker (par or inv) in the conversation
    '''
    speaker_dict = {
        'INV': [0 ,0 , 1],
        'PAR': [0, 1, 0],
        'padding': [1, 0, 0]
    }

    with open(transcription_filename, 'r') as f:
        content = f.read()
        content = content.split('\n')
        speakers = []

        for c in content:
            if 'INV' in c:
              speakers.append('INV')
            if 'PAR' in c:
              speakers.append('PAR')

        PAR_first_index = speakers.index('PAR')
        PAR_last_index = len(speakers) - speakers[::-1].index('PAR') - 1
        intervention_features = speakers[PAR_first_index:PAR_last_index]

    intervention_features = list(map(lambda x: speaker_dict[x], intervention_features))

    if len(intervention_features) > max_length:
        intervention_features = intervention_features[:max_length]
    else:
        pad_length = max_length - len(intervention_features)
        intervention_features =intervention_features+[speaker_dict['padding']]*pad_length

    return intervention_features


def get_spectogram_features(spectogram_filename):
    '''
    Spectogram features include MFCC which has been pregenerated for the audio file
    '''
    mel = np.load(spectogram_filename)
    # mel = feature_normalize(mel)
    mel = np.expand_dims(mel, axis=-1)
    return mel


def get_compare_features(compare_filename):
    compare_features = []
    with open(compare_filename, 'r') as file:
        content = csv.reader(file)
        for row in content:
            compare_features = row
    compare_features_floats = [float(item) for item in compare_features[1:-1]]
    return compare_features_floats


In [ ]:

def prepare_data(dataset_dir, config):
	'''
	Prepare all data
	'''
	################################## SUBJECTS ################################

	subject_files = sorted(glob.glob(os.path.join(dataset_dir, 'transcription/*/*.cha')))
	subjects = np.array(sorted(list(set([re.split('[/-]', i)[-2] for i in subject_files]))))


	######################################################################

	################################## INTERVENTION ####################################


	cc_files = sorted(glob.glob(os.path.join(dataset_dir, 'transcription/cc/*.cha')))
	all_speakers_cc = []
	for filename in cc_files:
		all_speakers_cc.append(get_intervention_features(filename, config.longest_speaker_length))
		# print(get_intervention_features(filename, longest_speaker_length))


	cd_files = sorted(glob.glob(os.path.join(dataset_dir, 'transcription/cd/*.cha')))
	all_speakers_cd = []
	for filename in cd_files:
		all_speakers_cd.append(get_intervention_features(filename, config.longest_speaker_length))
		# print(get_intervention_features(filename, longest_speaker_length))

	# b = [len(i) for i in all_speakers_cc]
	# print(len(b), np.min(b), np.mean(b), np.max(b))
	# a = [len(i) for i in all_speakers_cd]
	# print(len(a), np.min(a), np.mean(a), np.max(a), len([i for i in a if i>np.max(b)]))
	# exit()
	y_cc = np.zeros((len(all_speakers_cc), 2))
	y_cc[:,0] = 1

	y_cd = np.zeros((len(all_speakers_cd), 2))
	y_cd[:,1] = 1

	X_intervention = np.concatenate((all_speakers_cc, all_speakers_cd), axis=0).astype(np.float32)
	y_intervention = np.concatenate((y_cc, y_cd), axis=0).astype(np.float32)
	filenames_intervention = np.concatenate((cc_files, cd_files), axis=0)

	y_reg_cc = get_regression_values(os.path.join(dataset_dir, 'cc_meta_data.txt'))
	y_reg_cd = get_regression_values(os.path.join(dataset_dir, 'cd_meta_data.txt'))

	y_reg_intervention = np.concatenate((y_reg_cc, y_reg_cd), axis=0).astype(np.float32)
	X_reg_intervention = np.copy(X_intervention)
	################################## INTERVENTION ####################################

	################################## PAUSE ####################################

	cc_transcription_files = sorted(glob.glob(os.path.join(dataset_dir, 'transcription/cc/*.cha')))
	if dataset_dir == '../DementiaBank':
		cc_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cc/*.mp3')))
	else:
		cc_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cc/*.wav')))

	all_counts_cc = []
	for t_f, a_f in zip(cc_transcription_files, cc_audio_files):
		pause_features = get_pause_features(t_f, a_f)
		all_counts_cc.append(pause_features)

	cd_transcription_files = sorted(glob.glob(os.path.join(dataset_dir, 'transcription/cd/*.cha')))
	if dataset_dir == '../DementiaBank':
		cd_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cd/*.mp3')))
	else:
		cd_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cd/*.wav')))

	all_counts_cd = []
	for t_f, a_f in zip(cd_transcription_files, cd_audio_files):
		pause_features = get_pause_features(t_f, a_f)
		all_counts_cd.append(pause_features)

	X_pause = np.concatenate((all_counts_cc, all_counts_cd), axis=0).astype(np.float32)

	y_reg_cc = get_regression_values(os.path.join(dataset_dir, 'cc_meta_data.txt'))
	y_reg_cd = get_regression_values(os.path.join(dataset_dir, 'cd_meta_data.txt'))

	y_reg_pause = np.concatenate((y_reg_cc, y_reg_cd), axis=0).astype(np.float32)

	X_reg_pause = np.copy(X_pause)

	y_cc = np.zeros((len(all_counts_cc), 2))
	y_cc[:,0] = 1

	y_cd = np.zeros((len(all_counts_cd), 2))
	y_cd[:,1] = 1

	y_pause = np.concatenate((y_cc, y_cd), axis=0).astype(np.float32)
	filenames_pause = np.concatenate((cc_transcription_files, cd_transcription_files), axis=0)
	################################## PAUSE ####################################

	################################## COMPARE ####################################
	cc_files = sorted(glob.glob(os.path.join(dataset_dir, 'compare/cc/*.csv')))
	X_cc = np.array([get_compare_features(f) for f in cc_files])
	y_cc = np.zeros((X_cc.shape[0], 2))
	y_cc[:,0] = 1

	cd_files = sorted(glob.glob(os.path.join(dataset_dir, 'compare/cd/*.csv')))
	X_cd = np.array([get_compare_features(f) for f in cd_files])
	y_cd = np.zeros((X_cd.shape[0], 2))
	y_cd[:,1] = 1

	X_compare = np.concatenate((X_cc, X_cd), axis=0).astype(np.float32)
	y_compare = np.concatenate((y_cc, y_cd), axis=0).astype(np.float32)

	X_reg_compare = np.copy(X_compare)

	y_reg_cc = get_regression_values(os.path.join(dataset_dir, 'cc_meta_data.txt'))
	y_reg_cd = get_regression_values(os.path.join(dataset_dir, 'cd_meta_data.txt'))

	y_reg_compare = np.concatenate((y_reg_cc, y_reg_cd), axis=0).astype(np.float32)

	filenames_compare = np.concatenate((cc_files, cd_files), axis=0)
	################################## COMPARE ####################################

	assert np.array_equal(y_intervention, y_pause) and X_intervention.shape[0]==X_pause.shape[0], '~ Data streams are different ~'
	print('~ Data streams verified ~')

	y = y_intervention
	y_reg = y_reg_intervention

	X_intervention = X_intervention[p]
	X_pause = X_pause[p]
	# X_spec = X_spec[p]
	X_compare = X_compare[p]
	y = y[p]
	y_reg = y_reg[p]

	return {
		'intervention': X_intervention,
		'pause': X_pause,
		'compare': X_compare,
		'y_clf': y,
		'y_reg': y_reg,
		'subjects': subjects
	}


def read_wave(path):
	"""Reads a .wav file.
	Takes the path, and returns (PCM audio data, sample rate).
	"""
	with contextlib.closing(wave.open(path, 'rb')) as wf:
		num_channels = wf.getnchannels()
		assert num_channels == 1
		sample_width = wf.getsampwidth()
		assert sample_width == 2
		sample_rate = wf.getframerate()

		n_frames = wf.getnframes()
		data = wf.readframes(n_frames)

		converted = audioop.ratecv(data, sample_width, num_channels, sample_rate, 32000, None)
		return converted[0], 32000

class Frame(object):
	"""Represents a "frame" of audio data."""
	def __init__(self, bytes, timestamp, duration):
		self.bytes = bytes
		self.timestamp = timestamp
		self.duration = duration


def frame_generator(frame_duration_ms, audio, sample_rate):
	"""Generates audio frames from PCM audio data.
	Takes the desired frame duration in milliseconds, the PCM data, and
	the sample rate.
	Yields Frames of the requested duration.
	"""
	n = int(sample_rate * (frame_duration_ms / 1000.0) * 2)
	offset = 0
	timestamp = 0.0
	duration = (float(n) / sample_rate) / 2.0
	while offset + n < len(audio):
		yield Frame(audio[offset:offset + n], timestamp, duration)
		timestamp += duration
		offset += n


def vad_collector(sample_rate, frame_duration_ms,
                  padding_duration_ms, vad, frames):
	"""Filters out non-voiced audio frames.
	Given a webrtcvad.Vad and a source of audio frames, yields only
	the voiced audio.
	Uses a padded, sliding window algorithm over the audio frames.
	When more than 90% of the frames in the window are voiced (as
	reported by the VAD), the collector triggers and begins yielding
	audio frames. Then the collector waits until 90% of the frames in
	the window are unvoiced to detrigger.
	The window is padded at the front and back to provide a small
	amount of silence or the beginnings/endings of speech around the
	voiced frames.
	Arguments:
	sample_rate - The audio sample rate, in Hz.
	frame_duration_ms - The frame duration in milliseconds.
	padding_duration_ms - The amount to pad the window, in milliseconds.
	vad - An instance of webrtcvad.Vad.
	frames - a source of audio frames (sequence or generator).
	Returns: A generator that yields PCM audio data.
	"""
	num_padding_frames = int(padding_duration_ms / frame_duration_ms)
	# We use a deque for our sliding window/ring buffer.
	ring_buffer = collections.deque(maxlen=num_padding_frames)
	# We have two states: TRIGGERED and NOTTRIGGERED. We start in the
	# NOTTRIGGERED state.
	triggered = False

	voiced_frames = []
	silenced_frames = []
	for frame in frames:
		is_speech = vad.is_speech(frame.bytes, sample_rate)
		if is_speech:
			silenced_frames.append(5)
		else:
			silenced_frames.append(10)

	return silenced_frames


def get_pause_masks(file):
	frame_duration_ms = 30

	audio, sample_rate = read_wave(file)
	vad = webrtcvad.Vad()
	frames = frame_generator(frame_duration_ms, audio, sample_rate)
	frames = list(frames)
	segments = vad_collector(sample_rate, frame_duration_ms, 10, vad, frames)

	segments = np.asarray(segments)
	# segments = (segments - np.mean(segments))/np.std(segments)
	# print(segments)
	return segments

def prepare_data_new(dataset_dir, config):
	'''
	Prepare all data
	'''
	################################## SUBJECTS ################################
	subject_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cc/*.wav')) + glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cd/*.wav')))
	subjects = np.array(sorted(list(set([i.split('/')[-1][:-4] for i in subject_files]))))

	######################################################################

	################################## SILENCE MASK ####################################

	if 'ADReSS' in dataset_dir:
		cc_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cc/*.wav')))
	else:
		cc_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cc/*.mp3')))

	# can tweak this parameter for clipping the len of the mask
	max_len = 800

	all_counts_cc = []
	for a_f in cc_audio_files:
		pause_features = get_pause_masks(a_f)
		all_counts_cc.append(pause_features[:max_len])

	if dataset_dir == '../DementiaBank':
		cd_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cd/*.mp3')))
	else:
		cd_audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/cd/*.wav')))

	all_counts_cd = []
	for a_f in cd_audio_files:
		pause_features = get_pause_masks(a_f)
		all_counts_cd.append(pause_features[:max_len])

	all_counts_cd = np.asarray(all_counts_cd)
	all_counts_cc = np.asarray(all_counts_cc)
	# print("all_counts_cc : ", all_counts_cc.shape)
	# print("all_counts_cd : ", all_counts_cd.shape)

	X_pause = np.concatenate((all_counts_cc, all_counts_cd), axis=0).astype(np.float32)

	y_reg_cc = get_regression_values(os.path.join(dataset_dir, 'cc_meta_data.txt'))
	y_reg_cd = get_regression_values(os.path.join(dataset_dir, 'cd_meta_data.txt'))

	y_reg_pause = np.concatenate((y_reg_cc, y_reg_cd), axis=0).astype(np.float32)

	X_reg_pause = np.copy(X_pause)

	y_cc = np.zeros((len(all_counts_cc), 2))
	y_cc[:,0] = 1

	y_cd = np.zeros((len(all_counts_cd), 2))
	y_cd[:,1] = 1

	y_pause = np.concatenate((y_cc, y_cd), axis=0).astype(np.float32)

	print(X_pause.shape, y_pause.shape, y_reg_pause.shape)


	################################## SILENCE MASK ####################################
	return {
		# 'intervention': X_intervention,
		'silences': X_pause,
		# 'compare': X_compare,
		'y_clf': y_pause,
		'y_reg': y_reg_pause,
		'subjects': subjects
	}


def prepare_test_data(dataset_dir, config):

	'''
	Prepare test data
	'''
	################################## SUBJECTS ################################

	subject_files = sorted(glob.glob(os.path.join(dataset_dir, 'transcription/*.cha')))
	subjects = np.array(sorted(list(set([re.split('[/-]', i)[-2] for i in subject_files]))))

	######################################################################

	################################## INTERVENTION ####################################

	transcription_files = sorted(glob.glob(os.path.join(dataset_dir, 'transcription/*.cha')))
	all_speakers = []
	for filename in transcription_files:
		all_speakers.append(get_intervention_features(filename, config.longest_speaker_length))
	X_intervention = np.array(all_speakers)


	################################## PAUSE ####################################

	audio_files = sorted(glob.glob(os.path.join(dataset_dir, 'Full_wave_enhanced_audio/*.wav')))

	all_counts = []
	for t_f, a_f in zip(transcription_files, audio_files):
		pause_features = get_pause_features(t_f, a_f)
		all_counts.append(pause_features)
	X_pause = np.array(all_counts)


	################################## COMPARE ####################################
	compare_files = sorted(glob.glob(os.path.join(dataset_dir, 'compare/*.csv')))
	X_compare = np.array([get_compare_features(f) for f in compare_files])

	y = get_classification_values(os.path.join(dataset_dir, 'meta_data.txt'))
	y_reg = get_regression_values(os.path.join(dataset_dir, 'meta_data.txt'))


	assert X_intervention.shape[0]==X_pause.shape[0], '~ Data streams are different ~'
	print('~ Data streams verified ~')

	return {
		'intervention': X_intervention,
		'pause': X_pause,
		'compare': X_compare,
		'y_clf': y,
		'y_reg': y_reg,
		'subjects': subjects
	}

In [ ]:
# loading data, it is good to create either a function to crawl data or a class that contains the logic extraction logic

In [ ]:
# we have 2 types of files cha and wav


some cha readings

In [ ]:
# cha we load with
with open("001-0.cha", "r", encoding="utf-8") as f:
    data = f.read()

print(data)

@UTF8
@PID:	11312/a-00090320-0
@Window:	273_377_677_1118_-1_-1_2606_0_2606_0
@Begin
@Languages:	eng
@Participants:	INV Investigator, PAR Participant
@ID:	eng|Pitt|INV|||||Investigator|||
@ID:	eng|Pitt|PAR|57;|male|ProbableAD||Participant|18||
@Media:	001-0, audio
@Comment:	another audio testing file overlaps in background
@G:	Cookie
*INV:	this is the picture . 1360_2530
%mor:	pron|this-Dem-S1 aux|be-Fin-Ind-Pres-S3 det|the-Def-Art noun|picture .
%gra:	1|4|NSUBJ 2|4|COP 3|4|DET 4|0|ROOT 5|4|PUNCT
*PAR:	mhm . [+ exc] 2821_3211
%mor:	intj|mhm .
%gra:	1|0|ROOT 2|1|PUNCT
*INV:	just tell me everything that you see happening in that picture . 4022_6646
%mor:	adv|just verb|tell-Fin-Imp-S pron|I-Prs-Acc-S1 pron|everything-Tot-S1 pron|that-Rel pron|you-Prs-Nom-S2 verb|see-Fin-Ind-Pres-S2 verb|happen-Ger-S adp|in det|that-Def-Dem-Sing noun|picture .
%gra:	1|2|ADVMOD 2|0|ROOT 3|2|IOBJ 4|2|OBJ 5|8|OBJ 6|7|NSUBJ 7|4|ACL-RELCL 8|7|XCOMP 9|11|CASE 10|11|DET 11|8|OBL 12|2|PUNCT
*PAR:	+< alright .

In [2]:
utterances = []

with open("001-0.cha", "r", encoding="utf-8") as f:
    lines = f.readlines()

# for line in lines:
#     print(line.strip())

for line in lines:
    if line.startswith("*"):
        speaker, text = line.split(":", 1)
        utterances.append((speaker[1:], text.strip()))


best is with library

In [ ]:
reader = pylangacq.read_chat("001-0.cha")

print(reader.utterances())
reader.participants()
reader.words()

[Utterance(participant='INV', tokens=[...5 tokens], time_marks=Some((1360, 2530))), Utterance(participant='PAR', tokens=[...2 tokens], time_marks=Some((2821, 3211))), Utterance(participant='INV', tokens=[...12 tokens], time_marks=Some((4022, 6646))), Utterance(participant='PAR', tokens=[...2 tokens], time_marks=Some((6650, 6820))), Utterance(participant='PAR', tokens=[...10 tokens], time_marks=Some((7277, 12026))), Utterance(participant='PAR', tokens=[...12 tokens], time_marks=Some((12808, 18528))), Utterance(participant='PAR', tokens=[...14 tokens], time_marks=Some((19974, 24598))), Utterance(participant='PAR', tokens=[...10 tokens], time_marks=Some((25082, 29022))), Utterance(participant='PAR', tokens=[...13 tokens], time_marks=Some((30541, 34461))), Utterance(participant='PAR', tokens=[...9 tokens], time_marks=Some((35911, 38954))), Utterance(participant='PAR', tokens=[...7 tokens], time_marks=Some((39842, 42047))), Utterance(participant='PAR', tokens=[...7 tokens], time_marks=Some(

In [5]:
for utt in reader.utterances():
    print(utt.participant, utt.tokens)

INV [Token(word='this', pos='pron', mor='this-Dem-S1', gra=Gra(dep=1, head=4, rel='NSUBJ')), Token(word='is', pos='aux', mor='be-Fin-Ind-Pres-S3', gra=Gra(dep=2, head=4, rel='COP')), Token(word='the', pos='det', mor='the-Def-Art', gra=Gra(dep=3, head=4, rel='DET')), Token(word='picture', pos='noun', mor='picture', gra=Gra(dep=4, head=0, rel='ROOT')), Token(word='.', pos='', mor='.', gra=Gra(dep=5, head=4, rel='PUNCT'))]
PAR [Token(word='mhm', pos='intj', mor='mhm', gra=Gra(dep=1, head=0, rel='ROOT')), Token(word='.', pos='', mor='.', gra=Gra(dep=2, head=1, rel='PUNCT'))]
INV [Token(word='just', pos='adv', mor='just', gra=Gra(dep=1, head=2, rel='ADVMOD')), Token(word='tell', pos='verb', mor='tell-Fin-Imp-S', gra=Gra(dep=2, head=0, rel='ROOT')), Token(word='me', pos='pron', mor='I-Prs-Acc-S1', gra=Gra(dep=3, head=2, rel='IOBJ')), Token(word='everything', pos='pron', mor='everything-Tot-S1', gra=Gra(dep=4, head=2, rel='OBJ')), Token(word='that', pos='pron', mor='that-Rel', gra=Gra(dep=5, 

In [ ]:

rows = []

for utt in reader.utterances():
    text = " ".join(token.word for token in utt.tokens)

    rows.append({
        "speaker": utt.participant,
        "text": text,
        "time": utt.time_marks
    })


df = pd.DataFrame(rows)
print(df)

   speaker                                               text            time
0      INV                              this is the picture .    (1360, 2530)
1      PAR                                              mhm .    (2821, 3211)
2      INV  just tell me everything that you see happening...    (4022, 6646)
3      PAR                                          alright .    (6650, 6820)
4      PAR  there's a young boy that's getting a cookie jar .   (7277, 12026)
5      PAR  and he's in bad shape because the thing is fal...  (12808, 18528)
6      PAR  and in the picture the mother is washing dishe...  (19974, 24598)
7      PAR      and so the water is overflowing in the sink .  (25082, 29022)
8      PAR  and the dishes might fall over there if you do...  (30541, 34461)
9      PAR           and it's a picture of a kitchen window .  (35911, 38954)
10     PAR               and the curtains are very distinct .  (39842, 42047)
11     PAR                   but the water is still flowing .  (

wav file processing we can use librosa or the class above

In [14]:
get_pause_features("001-0.cha", "001-0.wav")

array([0.54353252, 0.        , 0.18117751, 1.26824254, 0.        ,
       0.        , 0.36235501, 0.        , 0.36235501, 0.        ,
       1.28273674])

In [ ]:
dataset_dir

def main(config):

	# Create save directories
	utils.create_directories(config)

	# Prepare and load the data
	if 'silences' in config.model_types:
		data = prepare_data_new(config.dataset_dir, config)
	else:
		data = prepare_data(config.dataset_dir, config)
	print(data)
	# return
# 	# Train the ensemble models
# 	if config.training_type == 'bagging':
# 		ensemble_trainer.bagging_ensemble_training(data, config)
# 	elif config.training_type == 'boosting':
# 		ensemble_trainer.boosted_ensemble_training(data, config)

# 	# Evaluate the model
# 	if 'silences' not in config.model_types:
# 		test_data = dataset.prepare_test_data(config.test_dataset_dir, config)
# 		evaluator.evaluate(data, test_data, config)

if __name__ == '__main__':
	main(config)

NameError: name 'config' is not defined

In [ ]:
import os
import numpy as np

def prepare_single_test_sample(cha_file, config, audio_file=None, compare_file=None):
    sample = {}

    # intervention features from .cha
    X_intervention = get_intervention_features(cha_file, config.longest_speaker_length)
    sample["intervention"] = np.expand_dims(np.asarray(X_intervention, dtype=np.float32), axis=0)

    # pause features from .cha + audio
    if audio_file is not None:
        X_pause = get_pause_features(cha_file, audio_file)
        sample["pause"] = np.expand_dims(np.asarray(X_pause, dtype=np.float32), axis=0)

    # compare features from csv
    if compare_file is not None:
        X_compare = get_compare_features(compare_file)
        sample["compare"] = np.expand_dims(np.asarray(X_compare, dtype=np.float32), axis=0)

    # subject name from filename
    subject = os.path.splitext(os.path.basename(cha_file))[0]
    sample["subjects"] = np.array([subject])

    return sample

In [ ]:
import numpy as np
import os

def prepare_single_cha(cha_file):
    x = get_intervention_features(cha_file, 32)
    pauses= get_pause_features(cha_file, "/content/002-0.wav")
    x = np.asarray(x, dtype=np.float32)
    x = np.expand_dims(x, axis=0)   # shape: (1, ...)

    subject = os.path.splitext(os.path.basename(cha_file))[0]
    return {
        "intervention": x,
        "subjects": np.array([subject])
    }

In [ ]:
# import utils

# config = utils.EasyDict({
# 	'task': 'classification',
# 	# 'task': 'regression',

# 	'uncertainty': False,

# 	# 'dataset_dir': '../DementiaBank'
# 	'dataset_dir': '../DementiaBank/0extra/ADReSS-IS2020-train/ADReSS-IS2020-data/train/',
# 	'test_dataset_dir': '../../alzheimers/ADReSS-IS2020-data/test',

# 	# 'model_dir': 'models/uncertainty_individual/1',
# 	# 'model_types': ['compare'],

# 	# 'model_dir': 'models/uncertainty_boosting_rmse/1',
# 	'model_dir': 'models/testing_silence/3_c',

# 	# 'model_types': [ 'compare', 'pause', 'intervention'],
# 	'model_types': [ 'silences'],

# 	'training_type': 'bagging',
# 	# 'training_type' :'boosting',

# 	'boosting_type': 'rmse',
# 	# 'boosting_type': 'stddev',


# 	'n_folds': 5,

# 	'dataset_split' :'full_dataset',
# 	# 'dataset_split' :'k_fold',
# 	'split_ratio': 0.8,

# 	'voting_type': 'hard_voting',
# 	# 'voting_type': 'soft_voting',
# 	# 'voting_type': 'learnt_voting',
# 	# 'voting_type': 'uncertainty_voting',


# 	'longest_speaker_length': 32,
# 	'n_pause_features': 11,
# 	'compare_features_size': 21,
# 	'split_reference': 'samples',

# 	'n_epochs': 2000,
# 	'batch_size': 24,
# 	'lr': 0.001,
# 	'verbose': 0
# })

In [ ]:
sample = prepare_single_cha("/content/002-0.cha")

print(sample["subjects"])
print(sample["intervention"].shape)
print()

AttributeError: module 'utils' has no attribute 'get_audio_length'

In [ ]:
sample["intervention"]

array([[[0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.]]], dtype=float32)